# 02 · Multiplicity model sets: what was trained and how the verdicts behave

Every table in this notebook is computed when it runs, from committed files: the deployed model sets
(`artifacts/models/*`, trained by `ml/training/train_all_variants.py`) and the feature table
(`artifacts/features/features.parquet`, 395 training and 99 held-out patients).

1. The near-optimal set for each marker and variant, with its cross-validated losses and skill over the base rate.
2. Held-out performance of every variant (fused, MRI only, age and sex only, fused with known IDH).
3. Out-of-fold predictions on the training patients, refitting each set member inside 5-fold cross-validation.
4. The stability verdict cut-offs evaluated on those out-of-fold predictions (the deployed values are documented
   defaults; this grid shows how they and the alternatives behave).
5. The 1p/19q decision threshold on out-of-fold predictions.
6. Held-out stability versus error, and the patients resolved by combining MRI with patient context.

In [1]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
ROOT=Path(".").resolve() if Path("artifacts").exists() else Path("../..").resolve()
sys.path[:0]=[str(ROOT/"backend"), str(ROOT/"ml"/"pipeline")]
from app.core.logging import configure_logging
configure_logging("WARNING")
from app.domain.registry import load_registry
import evaluate

registry=load_registry(ROOT/"artifacts", ROOT/"ml"/"configs"/"markers.yaml")
table=pd.read_parquet(ROOT/"artifacts"/"features"/"features.parquet")
train, test=table[table["split"]=="train"], table[table["split"]=="test"]
TARGET={"IDH": "idh", "1p19q": "codeletion_1p19q", "MGMT": "mgmt", "GRADE": "who_grade"}
def labels(marker, frame):
    y=frame[TARGET[marker]]
    return (y==4).astype(int) if marker=="GRADE" else y.astype(int)
print("training patients:", len(train), "| held-out patients:", len(test))

training patients:

395

| held-out patients:

99

## 1. The near-optimal sets

Each set keeps every candidate whose cross-validated log-loss is within ε (3%) of the best, or the best five when
fewer qualify. `skill` = improvement of the best member over always predicting the training base rate.

In [2]:
rows=[]
for (marker, variant), p in sorted(registry.predictors.items(), key=lambda kv: (kv[0][0].value, kv[0][1])):
    losses=[m.cv_log_loss for m in p.meta.models]
    rows.append({
        "marker": marker.value, "variant": variant, "members": len(losses),
        "families": ", ".join(sorted({m.family for m in p.meta.models})),
        "best CV log-loss": round(min(losses), 4), "worst member": round(max(losses), 4),
        "within 3% of best": sum(l<=min(losses)*1.03 for l in losses),
        "base-rate log-loss": p.meta.baseline_log_loss,
        "skill": None if p.meta.skill is None else f"{p.meta.skill:.1%}",
        "inputs": len(p.meta.feature_columns),
    })
pd.DataFrame(rows)

,marker,variant,members,families,best CV log-loss,worst member,within 3% of best,base-rate log-loss,skill,inputs
0,1p19q,clinical_only,10,logreg,0.1615,0.1655,10,0.1691,4.5%,2
1,1p19q,fused,5,"extratrees, logreg",0.1359,0.1416,2,0.1691,19.6%,266
2,1p19q,fused_given_idh,6,"extratrees, lightgbm, logreg",0.1281,0.1312,6,0.1691,24.2%,267
3,1p19q,imaging_only,5,extratrees,0.1357,0.1406,4,0.1691,19.8%,265
4,GRADE,clinical_only,10,logreg,0.3944,0.3963,10,0.4969,20.6%,2
5,GRADE,fused,5,"extratrees, lightgbm, logreg",0.1251,0.1295,3,0.4969,74.8%,266
6,GRADE,fused_given_idh,5,logreg,0.0995,0.1143,1,0.4969,80.0%,271
7,GRADE,imaging_only,5,"lightgbm, logreg",0.1239,0.1304,2,0.4969,75.1%,263
8,IDH,clinical_only,11,logreg,0.3050,0.3085,11,0.5053,39.6%,2
9,IDH,fused,5,"lightgbm, logreg",0.1993,0.2111,1,0.5053,60.6%,267


## 2. Held-out performance of every variant

AUC of the set's median probability on the 99 test patients, with a bootstrap 95% interval (1,000 resamples).
Comparing variants shows what each source adds: age and sex only, MRI only, both, and both plus a known IDH result.

In [3]:
results, per_patient=[], {}
for entry in evaluate.yaml.safe_load((ROOT/"ml"/"configs"/"markers.yaml").read_text())["markers"]:
    r, patients=evaluate.evaluate_set(entry["marker"], ROOT/"artifacts"/entry["artifact_dir"], test)
    results.append(r); per_patient[(r["marker"], r["variant"])]=patients
order=["clinical_only", "imaging_only", "fused", "fused_given_idh"]
pd.DataFrame([
    {"marker": r["marker"], "variant": r["variant"], "n": r["n_test"], "positives": r["n_positive"],
     "AUC": r["auc"], "95% CI": r.get("auc_ci95"), "balanced accuracy": r["balanced_accuracy"]}
    for r in sorted(results, key=lambda r: (r["marker"], order.index(r["variant"])))
])

,marker,variant,n,positives,AUC,95% CI,balanced accuracy
0,1p19q,clinical_only,80,2,0.6667,"[0.5063, 0.8101]",0.5000
1,1p19q,imaging_only,80,2,0.9038,"[0.7975, 0.9873]",0.5000
2,1p19q,fused,80,2,0.8974,"[0.7722, 1.0]",0.5000
3,1p19q,fused_given_idh,80,2,0.9744,"[0.9303, 1.0]",0.5000
4,GRADE,clinical_only,99,78,0.7546,"[0.6191, 0.878]",0.6300
5,GRADE,imaging_only,99,78,0.9963,"[0.9872, 1.0]",0.8810
6,GRADE,fused,99,78,0.9988,"[0.9943, 1.0]",0.9048
7,GRADE,fused_given_idh,99,78,0.9957,"[0.9842, 1.0]",0.9222
8,IDH,clinical_only,99,20,0.9139,"[0.8336, 0.9753]",0.7373
9,IDH,imaging_only,99,20,0.9633,"[0.925, 0.9888]",0.7873


## 3. Out-of-fold predictions on the training patients

Each member of the four fused sets is refitted inside 5-fold stratified cross-validation on the 395 training patients,
so every training patient gets a prediction from models that never saw it. Test patients are not used.

In [4]:
folds=StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
oof={}
for marker in ("IDH", "1p19q", "MGMT", "GRADE"):
    p=registry.predictors[(next(m for m in registry.predictors if m[0].value==marker and m[1]=="fused"))]
    rows=train[train[TARGET[marker]].notna()]
    X=rows.reindex(columns=p.meta.feature_columns).astype(float)
    y=labels(marker, rows).to_numpy()
    probs=np.zeros((len(p.models), len(y)))
    for train_idx, val_idx in folds.split(X, y):
        for i, model in enumerate(p.models):
            fitted=clone(model).fit(X.iloc[train_idx], y[train_idx])
            probs[i, val_idx]=fitted.predict_proba(X.iloc[val_idx])[:, 1]
    oof[marker]=(probs, y, p.meta.decision_threshold)
    print(f"{marker}: {probs.shape[0]} members x {probs.shape[1]} patients, out-of-fold AUC {roc_auc_score(y, np.median(probs, axis=0)):.3f}")

IDH: 5 members x 393 patients, out-of-fold AUC 0.963

1p19q: 5 members x 323 patients, out-of-fold AUC 0.851

MGMT: 24 members x 325 patients, out-of-fold AUC 0.536

GRADE: 5 members x 395 patients, out-of-fold AUC 0.974

## 4. Verdict cut-offs on out-of-fold predictions

For each pair of cut-offs (minimum agreement for Stable, maximum spread for Stable), the share of training patients
called Stable and the error rate of Stable calls versus the rest. A useful verdict makes Stable calls clearly more
reliable while still covering most patients. ★ marks the deployed defaults (0.85, 0.20).

In [5]:
def verdicts(probs, threshold, min_agree, max_iqr, arbitrary=0.60):
    votes=(probs>=threshold).mean(axis=0)
    agree=np.maximum(votes, 1-votes)
    iqr=np.percentile(probs, 75, axis=0)-np.percentile(probs, 25, axis=0)
    return np.where(agree<=arbitrary, "ARBITRARY", np.where((agree>=min_agree) & (iqr<=max_iqr), "STABLE", "CONTESTED"))

grid=[]
for marker, (probs, y, threshold) in oof.items():
    call=((probs>=threshold).mean(axis=0)>=0.5).astype(int)
    wrong=call!=y
    for min_agree in (0.8, 0.85, 0.9, 1.0):
        for max_iqr in (0.1, 0.15, 0.2, 0.25, 0.3):
            v=verdicts(probs, threshold, min_agree, max_iqr)
            stable=v=="STABLE"
            grid.append({
                "marker": marker, "min agreement": min_agree, "max spread": max_iqr,
                "deployed": "★" if (min_agree, max_iqr)==(0.85, 0.2) else "",
                "Stable share": f"{stable.mean():.0%}",
                "Stable error": f"{wrong[stable].mean():.1%}" if stable.any() else "-",
                "other error": f"{wrong[~stable].mean():.1%}" if (~stable).any() else "-",
            })
grid=pd.DataFrame(grid)
grid[grid["marker"].isin(["IDH", "GRADE"])]

,marker,min agreement,max spread,deployed,Stable share,Stable error,other error
0,IDH,0.80,0.10,,80%,4.5%,21.5%
1,IDH,0.80,0.15,,88%,4.6%,31.2%
2,IDH,0.80,0.20,,93%,5.2%,42.9%
3,IDH,0.80,0.25,,94%,6.2%,36.4%
4,IDH,0.80,0.30,,95%,6.4%,38.9%
5,IDH,0.85,0.10,,79%,3.9%,23.2%
6,IDH,0.85,0.15,,87%,4.1%,32.7%
7,IDH,0.85,0.20,★,92%,4.7%,43.8%
8,IDH,0.85,0.25,,93%,5.2%,41.4%
9,IDH,0.85,0.30,,93%,5.4%,42.3%


In [6]:
grid[grid["marker"].isin(["1p19q", "MGMT"])]

,marker,min agreement,max spread,deployed,Stable share,Stable error,other error
20,1p19q,0.80,0.10,,97%,2.5%,44.4%
21,1p19q,0.80,0.15,,99%,3.1%,50.0%
22,1p19q,0.80,0.20,,100%,3.7%,0.0%
23,1p19q,0.80,0.25,,100%,3.7%,0.0%
24,1p19q,0.80,0.30,,100%,3.7%,0.0%
25,1p19q,0.85,0.10,,97%,2.5%,44.4%
26,1p19q,0.85,0.15,,99%,3.1%,50.0%
27,1p19q,0.85,0.20,★,100%,3.7%,0.0%
28,1p19q,0.85,0.25,,100%,3.7%,0.0%
29,1p19q,0.85,0.30,,100%,3.7%,0.0%


## 5. The 1p/19q decision threshold

Co-deletion is rare (about 4% of training patients), so calibrated probabilities rarely reach 0.5 and the default
threshold makes no positive calls. Precision and recall of the set's majority call at lower thresholds, on the
out-of-fold predictions only (choosing a threshold on the 2 positive test patients would be meaningless).

In [7]:
probs, y, _=oof["1p19q"]
print("codeleted training patients:", int(y.sum()), "of", len(y))
rows=[]
for t in (0.5, 0.3, 0.2, 0.15, 0.1, 0.05):
    call=((probs>=t).mean(axis=0)>=0.5).astype(int)
    tp=int(((call==1) & (y==1)).sum()); fp=int(((call==1) & (y==0)).sum())
    rows.append({"threshold": t, "positive calls": int(call.sum()), "true positives": tp, "false positives": fp,
                 "recall": f"{tp/max(int(y.sum()),1):.0%}", "precision": f"{tp/max(tp+fp,1):.0%}"})
pd.DataFrame(rows)

codeleted training patients:

13

of

323

,threshold,positive calls,true positives,false positives,recall,precision
0,0.50,1,1,0,8%,100%
1,0.30,3,1,2,8%,33%
2,0.20,5,1,4,8%,20%
3,0.15,13,5,8,38%,38%
4,0.10,28,5,23,38%,18%
5,0.05,62,10,52,77%,16%


## 6. Held-out check: stability versus error, and resolved by combining

On the 99 test patients with the deployed cut-offs: the error rate within each verdict, and the patients whose MRI-only
set was not Stable but whose combined set was.

In [8]:
display(pd.DataFrame([
    {"marker": r["marker"], "variant": r["variant"],
     **{f"{v} n / error": f"{r['by_verdict'][v]['n']} / {r['by_verdict'][v]['error_rate']}" for v in ("STABLE", "CONTESTED", "ARBITRARY")}}
    for r in results if r["variant"]=="fused"
]))
pd.DataFrame([
    {"marker": m, **evaluate.resolved_by_combining(per_patient[(m, "fused")], per_patient[(m, "imaging_only")])}
    for m in ("IDH", "1p19q", "MGMT", "GRADE")
])

,marker,variant,STABLE n / error,CONTESTED n / error,ARBITRARY n / error
0,IDH,fused,90 / 0.0222,7 / 0.4286,2 / 0.5
1,1p19q,fused,80 / 0.025,0 / None,0 / None
2,MGMT,fused,84 / 0.2976,0 / None,0 / None
3,GRADE,fused,90 / 0.0111,7 / 0.1429,2 / 1.0


,marker,n_test,imaging_only_not_stable,resolved_by_combining,fused_accuracy_on_resolved,stable_only_without_context
0,IDH,99,6,3,1.0,6
1,1p19q,80,0,0,NaN,0
2,MGMT,84,0,0,NaN,0
3,GRADE,99,7,2,1.0,4
